<a href="https://colab.research.google.com/github/uday-andotra/WQU/blob/main/MScFE_620_DP_GWP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# =====================================================================
# GLOBAL PARAMETERS
# =====================================================================
S0 = 80
K_base = 80
r = 0.055
T = 0.25  # 3 months

# Monte Carlo setup
N_paths = 50000
N_steps = 250

# Heston Parameters (Member A)
v0 = 0.032
kappa = 1.85
theta = 0.045
sigma_v = 0.35

# Merton Parameters (Member B)
sigma_m = 0.35  # Base volatility
mu_J = -0.5
delta_J = 0.22

np.random.seed(42) # For reproducible results

# Calculate global dt
dt = T / N_steps

# =====================================================================
# CORE SIMULATION FUNCTIONS
# =====================================================================

def heston_paths(S0, K, r, T, v0, kappa, theta, sigma_v, rho, n_paths, n_steps):
    """Simulates Heston paths perfectly matching Member A's math"""
    S_paths = np.zeros((n_steps + 1, n_paths))
    v_paths = np.zeros((n_steps + 1, n_paths))
    S_paths[0] = S0
    v_paths[0] = v0


    for t in range(1, n_steps + 1):
        z1 = np.random.normal(size=n_paths)
        z2 = np.random.normal(size=n_paths)
        z2 = rho * z1 + np.sqrt(1 - rho**2) * z2

        # Update Variance
        v_paths[t] = np.abs(v_paths[t-1] + kappa * (theta - v_paths[t-1]) * dt + sigma_v * np.sqrt(v_paths[t-1] * dt) * z2)

        # Update Stock using the newly calculated Variance (Member A's exact sequence)
        S_paths[t] = S_paths[t-1] * np.exp((r - 0.5 * v_paths[t]) * dt + np.sqrt(v_paths[t] * dt) * z1)

    call_price = np.exp(-r * T) * np.mean(np.maximum(S_paths[-1] - K, 0))
    put_price = np.exp(-r * T) * np.mean(np.maximum(K - S_paths[-1], 0))

    return call_price, put_price, S_paths

def merton_paths(S0, K, r, T, sigma, mu, delta, lamb, n_paths, n_steps):
    """Simulates Merton paths using local dt to prevent global poisoning"""
    S_paths = np.zeros((n_steps + 1, n_paths))
    S_paths[0] = S0


    m = np.exp(mu + 0.5 * delta**2) - 1
    drift = r - lamb * m - 0.5 * sigma**2

    for t in range(1, n_steps + 1):
        Z = np.random.standard_normal(n_paths)
        N = np.random.poisson(lamb * dt, n_paths)
        J = N * mu + np.sqrt(N) * delta * np.random.standard_normal(n_paths)

        S_paths[t] = S_paths[t-1] * np.exp(drift * dt + sigma * np.sqrt(dt) * Z + J)

    call_price = np.exp(-r * T) * np.mean(np.maximum(S_paths[-1] - K, 0))
    put_price = np.exp(-r * T) * np.mean(np.maximum(K - S_paths[-1], 0))

    return call_price, put_price, S_paths

def lsmc_american_call(paths, K, r, dt):
    """Longstaff-Schwartz Monte Carlo algorithm for American Call"""
    steps, sims = paths.shape
    discount = np.exp(-r * dt)
    cashflow = np.maximum(paths[-1] - K, 0)

    for t in range(steps - 2, 0, -1):
        itm = paths[t] > K
        if np.sum(itm) > 0:
            X = paths[t, itm]
            Y = cashflow[itm] * discount
            poly = np.polyfit(X, Y, 2)
            continuation_value = np.polyval(poly, X)
            exercise_value = X - K
            exercise = exercise_value > continuation_value
            cashflow[itm] = np.where(exercise, exercise_value, Y)
            cashflow[~itm] = cashflow[~itm] * discount
        else:
            cashflow = cashflow * discount

    return np.mean(cashflow * discount)

# =====================================================================
# TEAM MEMBER A: HESTON MODEL (Questions 5 - 7)
# =====================================================================
print("================ TEAM MEMBER A ================")

# Question 5: rho = -0.30
c_q5, p_q5, h_paths_q5 = heston_paths(S0, K_base, r, T, v0, kappa, theta, sigma_v, -0.30, N_paths, N_steps)
print("\n--- Question 5: Heston (rho = -0.30) ---")
print(pd.DataFrame([["Heston Call", c_q5], ["Heston Put", p_q5]], columns=["Option", "Price"]).round(2).to_markdown(index=False))

# Question 6: rho = -0.70
c_q6, p_q6, h_paths_q6 = heston_paths(S0, K_base, r, T, v0, kappa, theta, sigma_v, -0.70, N_paths, N_steps)
print("\n--- Question 6: Heston (rho = -0.70) ---")
print(pd.DataFrame([["Heston Call", c_q6], ["Heston Put", p_q6]], columns=["Option", "Price"]).round(2).to_markdown(index=False))

# Question 7: Heston Delta & Gamma (Numerical Approximation using Q6 params)
eps = 0.80 # 1% bump
c_up, p_up, _ = heston_paths(S0 + eps, K_base, r, T, v0, kappa, theta, sigma_v, -0.70, N_paths, N_steps)
c_dn, p_dn, _ = heston_paths(S0 - eps, K_base, r, T, v0, kappa, theta, sigma_v, -0.70, N_paths, N_steps)

delta_c_h = (c_up - c_dn) / (2 * eps)
gamma_c_h = (c_up - 2 * c_q6 + c_dn) / (eps**2)
delta_p_h = (p_up - p_dn) / (2 * eps)
gamma_p_h = (p_up - 2 * p_q6 + p_dn) / (eps**2)

print("\n--- Question 7: Heston Greeks ---")
print(pd.DataFrame([
    ["Call", delta_c_h, gamma_c_h],
    ["Put", delta_p_h, gamma_p_h]
], columns=["Option", "Delta", "Gamma"]).round(4).to_markdown(index=False))


# =====================================================================
# TEAM MEMBER B: MERTON MODEL (Questions 8 - 10)
# =====================================================================
print("\n\n================ TEAM MEMBER B ================")

# Question 8: lambda = 0.75
c_q8, p_q8, m_paths_q8 = merton_paths(S0, K_base, r, T, sigma_m, mu_J, delta_J, 0.75, N_paths, N_steps)
print("\n--- Question 8: Merton (lambda = 0.75) ---")
print(pd.DataFrame([["Merton Call", c_q8], ["Merton Put", p_q8]], columns=["Option", "Price"]).round(2).to_markdown(index=False))

# Question 9: lambda = 0.25
c_q9, p_q9, m_paths_q9 = merton_paths(S0, K_base, r, T, sigma_m, mu_J, delta_J, 0.25, N_paths, N_steps)
print("\n--- Question 9: Merton (lambda = 0.25) ---")
print(pd.DataFrame([["Merton Call", c_q9], ["Merton Put", p_q9]], columns=["Option", "Price"]).round(2).to_markdown(index=False))

# Question 10: Merton Delta & Gamma (Numerical Approximation using Q8 params)
c_up_m, p_up_m, _ = merton_paths(S0 + eps, K_base, r, T, sigma_m, mu_J, delta_J, 0.75, N_paths, N_steps)
c_dn_m, p_dn_m, _ = merton_paths(S0 - eps, K_base, r, T, sigma_m, mu_J, delta_J, 0.75, N_paths, N_steps)

delta_c_m = (c_up_m - c_dn_m) / (2 * eps)
gamma_c_m = (c_up_m - 2 * c_q8 + c_dn_m) / (eps**2)
delta_p_m = (p_up_m - 2 * p_q8 + p_dn_m) / (eps**2)

print("\n--- Question 10: Merton Greeks ---")
print(pd.DataFrame([
    ["Call", delta_c_m, gamma_c_m],
    ["Put", delta_p_m, gamma_p_m]
], columns=["Option", "Delta", "Gamma"]).round(4).to_markdown(index=False))


# =====================================================================
# TEAM MEMBER C: VALIDATION & ADVANCED (Questions 11 - 15)
# =====================================================================
print("\n\n================ TEAM MEMBER C ================")

# Question 11: Put-Call Parity
rhs_parity = S0 - K_base * np.exp(-r * T)
results_q11 = [
    ["Heston (Q5)", c_q5, p_q5, c_q5 - p_q5, rhs_parity],
    ["Heston (Q6)", c_q6, p_q6, c_q6 - p_q6, rhs_parity],
    ["Merton (Q8)", c_q8, p_q8, c_q8 - p_q8, rhs_parity],
    ["Merton (Q9)", c_q9, p_q9, c_q9 - p_q9, rhs_parity]
]
df_q11 = pd.DataFrame(results_q11, columns=["Model", "Call", "Put", "LHS (C - P)", "RHS (S0 - K*e^-rT)"])
df_q11["Difference"] = df_q11["LHS (C - P)"] - df_q11["RHS (S0 - K*e^-rT)"]
print("\n--- Question 11: Put-Call Parity Check ---")
print(df_q11.round(2).to_markdown(index=False))

# Question 12: Strike Iterations
moneyness = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
strikes = [S0 / m for m in moneyness]
results_q12 = []

for m, k in zip(moneyness, strikes):
    c_h, _, _ = heston_paths(S0, k, r, T, v0, kappa, theta, sigma_v, -0.30, N_paths, N_steps)
    c_m, _, _ = merton_paths(S0, k, r, T, sigma_m, mu_J, delta_J, 0.75, N_paths, N_steps)
    results_q12.append([m, k, c_h, c_m])

print("\n--- Question 12: Strike Iterations (Calls) ---")
print(pd.DataFrame(results_q12, columns=["Moneyness", "Strike", "Heston (Q5)", "Merton (Q8)"]).round(2).to_markdown(index=False))

# Question 13: American Call Option
amer_call_q5 = lsmc_american_call(h_paths_q5, K_base, r, dt)
print("\n--- Question 13: American Call (Heston Q5 parameters) ---")
print(pd.DataFrame([
    ["European Call", c_q5],
    ["American Call", amer_call_q5]
], columns=["Option Type", "Price"]).round(2).to_markdown(index=False))

# Question 14: Up-and-In Call (Heston Q6)
H_call = 95
K_call = 95
barrier_breached_c = np.max(h_paths_q6, axis=0) >= H_call
payoffs_cui = np.maximum(h_paths_q6[-1] - K_call, 0) * barrier_breached_c
cui_price = np.exp(-r * T) * np.mean(payoffs_cui)

# Get standard Euro Call at K=95 to compare
euro_call_95, _, _ = heston_paths(S0, K_call, r, T, v0, kappa, theta, sigma_v, -0.70, N_paths, N_steps)

print("\n--- Question 14: Up-and-In Call ---")
print(pd.DataFrame([
    ["European Call (K=95)", euro_call_95],
    ["Up-and-In Call (K=95, H=95)", cui_price]
], columns=["Option Type", "Price"]).round(2).to_markdown(index=False))

# Question 15: Down-and-In Put (Merton Q8)
H_put = 65
K_put = 65
barrier_breached_p = np.min(m_paths_q8, axis=0) <= H_put
payoffs_pdi = np.maximum(K_put - m_paths_q8[-1], 0) * barrier_breached_p
pdi_price = np.exp(-r * T) * np.mean(payoffs_pdi)

# Get standard Euro Put at K=65 to compare
_, euro_put_65, _ = merton_paths(S0, K_put, r, T, sigma_m, mu_J, delta_J, 0.75, N_paths, N_steps)

print("\n--- Question 15: Down-and-In Put ---")
print(pd.DataFrame([
    ["European Put (K=65)", euro_put_65],
    ["Down-and-In Put (K=65, H=65)", pdi_price]
], columns=["Option Type", "Price"]).round(2).to_markdown(index=False))

from google.colab import output
output.no_vertical_scroll()
print("")
print("")
print("")
print("")


================ TEAM MEMBER A ================

--- Question 5: Heston (rho = -0.30) ---
| Option      |   Price |
|:------------|--------:|
| Heston Call |    2.85 |
| Heston Put  |    2.82 |

--- Question 6: Heston (rho = -0.70) ---
| Option      |   Price |
|:------------|--------:|
| Heston Call |    2.11 |
| Heston Put  |    3.47 |

--- Question 7: Heston Greeks ---
| Option   |   Delta |   Gamma |
|:---------|--------:|--------:|
| Call     |  0.4548 |  0.0528 |
| Put      | -0.4362 |  0.0524 |


================ TEAM MEMBER B ================

--- Question 8: Merton (lambda = 0.75) ---
| Option      |   Price |
|:------------|--------:|
| Merton Call |    8.22 |
| Merton Put  |    7.28 |

--- Question 9: Merton (lambda = 0.25) ---
| Option      |   Price |
|:------------|--------:|
| Merton Call |    6.81 |
| Merton Put  |    5.72 |

--- Question 10: Merton Greeks ---
| Option   |   Delta |   Gamma |
|:---------|--------:|--------:|
| Call     |  0.604  |  0.3483 |
| Put      |

<IPython.core.display.Javascript object>